In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor

RAW = r"C:\Users\reinh\Downloads\Projects\ab-uplift-hillstrom\Kevin_Hillstrom_MineThatData.csv"
OUT = r"C:\Users\reinh\Downloads\Projects\ab-uplift-hillstrom\abuh\Outputs"

In [2]:
df = pd.read_csv(RAW)
df["email_sent"] = (df["segment"] != "No E-Mail").astype(int)

print(df["segment"].value_counts())

segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64


In [3]:
# Check pre-treatment balance across arms using Standardised Mean Differences (|SMD| < 0.1 = balanced)
num_cols = ["recency", "history"]
cat_cols = ["history_segment", "mens", "womens", "zip_code", "newbie", "channel"]

X_bal = pd.get_dummies(df[num_cols + cat_cols], columns=cat_cols, dtype=float)
ctrl = X_bal[df["segment"] == "No E-Mail"]

for arm in ["Mens E-Mail", "Womens E-Mail"]:
    trt = X_bal[df["segment"] == arm]
    smd = (trt.mean() - ctrl.mean()) / np.sqrt((trt.var() + ctrl.var()) / 2)
    print(f"{arm} max |SMD|: {smd.abs().max():.3f}")

# Summary metrics by arm
outcomes_by_arm = df.groupby("segment").agg(
    customers=("visit", "size"),
    visit_rate=("visit", "mean"),
    conversion_rate=("conversion", "mean"),
    avg_spend=("spend", "mean"),
).round(4)

print("\n", outcomes_by_arm)
outcomes_by_arm.to_csv(f"{OUT}/m1_outcomes_by_arm.csv")

Mens E-Mail max |SMD|: 0.016
Womens E-Mail max |SMD|: 0.014

                customers  visit_rate  conversion_rate  avg_spend
segment                                                         
Mens E-Mail        21307      0.1828           0.0125     1.4226
No E-Mail          21306      0.1062           0.0057     0.6528
Womens E-Mail      21387      0.1514           0.0088     1.0772


In [4]:
def compare_groups(a, b, label_a, label_b):
    rows = []
    for y in ["visit", "conversion", "spend"]:
        diff = a[y].mean() - b[y].mean()
        se = np.sqrt(a[y].var() / len(a) + b[y].var() / len(b))
        _, p = stats.ttest_ind(a[y], b[y], equal_var=False)
        rows.append({
            "outcome": y,
            f"{label_a}_mean": a[y].mean(),
            f"{label_b}_mean": b[y].mean(),
            "difference": diff,
            "ci_low": diff - 1.96 * se,
            "ci_high": diff + 1.96 * se,
            "p_value": round(p, 5),
        })
    return pd.DataFrame(rows)

ctl = df[df["segment"] == "No E-Mail"]
emailed = df[df["email_sent"] == 1]
mens = df[df["segment"] == "Mens E-Mail"]
womens = df[df["segment"] == "Womens E-Mail"]

overall = compare_groups(emailed, ctl, "email", "control")
mens_vs_womens = compare_groups(mens, womens, "mens", "womens")

print("=== Overall: Emailed vs Control ===")
print(overall.round(4).to_string(index=False))

print("\n=== Direct Causal Comparison: Men's vs Women's Email ===")
print(mens_vs_womens.round(4).to_string(index=False))

overall.to_csv(f"{OUT}/m2_overall_test.csv", index=False)
mens_vs_womens.to_csv(f"{OUT}/m3_mens_vs_womens.csv", index=False)

=== Overall: Emailed vs Control ===
   outcome  email_mean  control_mean  difference  ci_low  ci_high  p_value
     visit      0.1670        0.1062      0.0609  0.0554   0.0663      0.0
conversion      0.0107        0.0057      0.0050  0.0035   0.0064      0.0
     spend      1.2496        0.6528      0.5968  0.3762   0.8174      0.0

=== Direct Causal Comparison: Men's vs Women's Email ===
   outcome  mens_mean  womens_mean  difference  ci_low  ci_high  p_value
     visit     0.1828       0.1514      0.0314  0.0243   0.0384   0.0000
conversion     0.0125       0.0088      0.0037  0.0017   0.0056   0.0002
     spend     1.4226       1.0772      0.3454  0.0326   0.6583   0.0305


In [5]:
FEATURES = ["recency", "history", "history_segment", "mens", "womens", "zip_code", "newbie", "channel"]
X = pd.get_dummies(df[FEATURES], columns=["history_segment", "zip_code", "channel"], dtype=float)

def crossfit_uplift(outcome, is_classifier):
    Model = HistGradientBoostingClassifier if is_classifier else HistGradientBoostingRegressor
    params = dict(max_iter=200, learning_rate=0.05, min_samples_leaf=50, random_state=42)
    scores = np.zeros(len(df))
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    for tr, te in skf.split(X, df["segment"]):
        tr_e = tr[df.iloc[tr]["email_sent"] == 1]
        tr_c = tr[df.iloc[tr]["email_sent"] == 0]

        m1 = Model(**params).fit(X.iloc[tr_e], df.iloc[tr_e][outcome])
        m0 = Model(**params).fit(X.iloc[tr_c], df.iloc[tr_c][outcome])

        p1 = m1.predict_proba(X.iloc[te])[:, 1] if is_classifier else m1.predict(X.iloc[te])
        p0 = m0.predict_proba(X.iloc[te])[:, 1] if is_classifier else m0.predict(X.iloc[te])
        scores[te] = p1 - p0

    return scores

df["uplift_conv_cf"] = crossfit_uplift("conversion", is_classifier=True)
df["uplift_spend_cf"] = crossfit_uplift("spend", is_classifier=False)
df["random_score"] = np.random.default_rng(42).random(len(df))

print("Avg predicted conversion uplift:", round(df["uplift_conv_cf"].mean(), 5))
print("Avg predicted spend uplift:     ", round(df["uplift_spend_cf"].mean(), 4))

Avg predicted conversion uplift: 0.00485
Avg predicted spend uplift:      0.5714


In [6]:
def decile_uplift(outcome, rank_col):
    grp = pd.qcut(df[rank_col].rank(method="first"), 10, labels=False) + 1
    rows = []
    for d, g in df.groupby(grp):
        e = g.loc[g["email_sent"] == 1, outcome]
        c = g.loc[g["email_sent"] == 0, outcome]
        rows.append({"decile": d, "customers": len(g), "actual_uplift": e.mean() - c.mean()})
    return pd.DataFrame(rows)

spend_cf = decile_uplift("spend", "uplift_spend_cf")
print("=== Spend Uplift by Predicted Decile (10 = Highest) ===")
print(spend_cf.round(4).to_string(index=False))
spend_cf.to_csv(f"{OUT}/m4_cf_spend_deciles.csv", index=False)

=== Spend Uplift by Predicted Decile (10 = Highest) ===
 decile  customers  actual_uplift
      1       6400         0.9729
      2       6400         0.9962
      3       6400         0.3237
      4       6400         0.1340
      5       6400         0.3002
      6       6400         0.3093
      7       6400         0.9528
      8       6400        -0.0919
      9       6400         1.0543
     10       6400         1.0304


In [7]:
RANKINGS = {
    "Spend model": "uplift_spend_cf",
    "Conversion model": "uplift_conv_cf",
    "Past spend": "history",
    "Random": "random_score",
}

fractions = np.round(np.arange(0.1, 1.01, 0.1), 1)
curves = []

for name, col in RANKINGS.items():
    ranked = df.sort_values(col, ascending=False)
    for f in fractions:
        top = ranked.iloc[: int(round(f * len(ranked)))]
        e = top.loc[top["email_sent"] == 1, "spend"]
        c = top.loc[top["email_sent"] == 0, "spend"]
        uplift = e.mean() - c.mean()
        curves.append({
            "ranking": name,
            "fraction": f,
            "uplift_per_customer": uplift,
            "net_at_0.10": len(df) * f * (uplift - 0.10),
            "net_at_0.25": len(df) * f * (uplift - 0.25),
        })

curves_df = pd.DataFrame(curves)
print(curves_df.pivot(index="fraction", columns="ranking", values="uplift_per_customer").round(4))

curves_df.to_csv(f"{OUT}/m5_targeting_curves.csv", index=False)
df[["uplift_conv_cf", "uplift_spend_cf", "random_score", "history", "email_sent", "conversion", "spend"]].to_csv(
    f"{OUT}/customer_scores_crossfit.csv", index=False
)
print(f"\nSaved outputs to: {OUT}")

ranking   Conversion model  Past spend  Random  Spend model
fraction                                                   
0.1                 0.8605      1.3592  0.2708       1.0304
0.2                 0.8563      1.1434  0.4110       1.0423
0.3                 0.7200      0.8710  0.7263       0.6634
0.4                 0.8523      0.6002  0.6248       0.7346
0.5                 0.6881      0.6955  0.4931       0.6502
0.6                 0.5926      0.6111  0.5415       0.5917
0.7                 0.5726      0.5718  0.6136       0.5263
0.8                 0.5123      0.5587  0.5468       0.5000
0.9                 0.5679      0.5996  0.5957       0.5551
1.0                 0.5968      0.5968  0.5968       0.5968

Saved outputs to: C:\Users\reinh\Downloads\Projects\ab-uplift-hillstrom\abuh\Outputs
